# I convert Hattie Chung's data set into an .RData file.

In [1]:
from preamble import *

cpu


In [ ]:
# # Global parameters:
number_of_cycles = 500 # how many passes through the training data we go through
number_of_groups = 1 # divide the data set into smaller ones, to make fitting easier.
locations_per_group = 200 # how many locations to observe per group
number_of_locations = number_of_groups * locations_per_group # total locations
number_of_simulations = 50 # for synthetic data, how many optimisation to average over
steps_per_batch = 5
dims = 2  # 2D spatial
p =  3 # how many features 

In [41]:
def isolate_gene_values (adata, gene_name):
    gene_values = pd.DataFrame(adata[:, gene_name].X.toarray(), columns=[gene_name], index=adata.obs_names)
    return gene_values

head=1500
adata = sc.read_h5ad('ovary_Puck_230517_39.h5ad')
coordinates = pd.DataFrame(adata.obsm["spatial"], columns=['x', 'y'], index=adata.obs_names)
df = pd.concat([
    coordinates,
    isolate_gene_values(adata, "Serpine2"),
    isolate_gene_values(adata, "Tagln"),
    isolate_gene_values(adata, "Acta2"),
    isolate_gene_values(adata, "Mgp"),
    isolate_gene_values(adata, "S100a6"),
    isolate_gene_values(adata, "Col1a2"),
    isolate_gene_values(adata, "Nr5a2"),
    isolate_gene_values(adata, "Inhba"),
    isolate_gene_values(adata, "Tpm2"),
    isolate_gene_values(adata, "Tdrd5")
], axis=1)
df = df.sample(frac=1)
df = df.head(head)
df['x'] = df['x'] / df['x'].median()
df['y'] = df['y'] / df['y'].median()

# Normalize all gene columns by dividing by 1000
genes_to_include = ["Serpine2", "Tagln", "Acta2", "Mgp", "S100a6", "Col1a2", "Nr5a2", "Inhba", "Tpm2", "Tdrd5"]
for gene in genes_to_include:
    df[gene] = df[gene] / 1000
df = df.reset_index(drop=True)
# Create a dictionary mapping gene names to numbers
gene_mapping = {
    "Serpine2": 1,
    "Tagln": 2,
    "Acta2": 3,
    "Mgp": 4,
    "S100a6": 5,
    "Col1a2": 6,
    "Nr5a2": 7,
    "Inhba": 8,
    "Tpm2": 9,
    "Tdrd5": 10
}
df

,x,y,Serpine2,Tagln,Acta2,Mgp,S100a6,Col1a2,Nr5a2,Inhba,Tpm2,Tdrd5
0,1.044411,1.107810,0.000000,0.000000,0.000000,0.000000,0.000000,0.124688,0.124688,0.000000,0.000000,0.000000
1,1.215361,1.164496,0.000000,0.027174,0.054348,0.217391,0.000000,0.217391,0.081522,0.000000,0.054348,0.000000
2,0.776470,1.293055,0.000000,0.000000,0.000000,0.000000,0.000000,0.268817,0.000000,0.000000,0.000000,0.000000
3,0.648007,0.755464,0.000000,0.000000,0.000000,1.953125,0.390625,0.000000,0.000000,0.000000,0.000000,0.000000
4,0.948926,0.666519,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
1495,0.659424,0.938788,0.000000,0.275482,0.000000,0.826446,0.000000,0.275482,0.000000,0.000000,0.275482,0.000000
1496,0.703216,1.139663,0.161812,0.000000,0.000000,0.000000,0.485437,0.000000,0.000000,0.000000,0.000000,0.000000
1497,1.003130,1.071929,0.209096,0.000000,0.000000,0.000000,0.052274,0.000000,0.156822,0.261370,0.000000,0.052274
1498,0.941627,1.057998,0.350593,0.000000,0.000000,0.026969,0.000000,0.000000,0.053937,0.188781,0.000000,0.000000


In [42]:
df = pd.melt(df, id_vars=['x', 'y'], var_name='Gene', value_name='Expression')
df = df[['Expression', 'x','y','Gene']]
df

,Expression,x,y,Gene
0,0.000000,1.044411,1.107810,Serpine2
1,0.000000,1.215361,1.164496,Serpine2
2,0.000000,0.776470,1.293055,Serpine2
3,0.000000,0.648007,0.755464,Serpine2
4,0.000000,0.948926,0.666519,Serpine2
...,...,...,...,...
14995,0.000000,0.659424,0.938788,Tdrd5
14996,0.000000,0.703216,1.139663,Tdrd5
14997,0.052274,1.003130,1.071929,Tdrd5
14998,0.000000,0.941627,1.057998,Tdrd5


In [43]:
# Replace the gene names with numbers
df['Gene'] = df['Gene'].map(gene_mapping)
df

,Expression,x,y,Gene
0,0.000000,1.044411,1.107810,1
1,0.000000,1.215361,1.164496,1
2,0.000000,0.776470,1.293055,1
3,0.000000,0.648007,0.755464,1
4,0.000000,0.948926,0.666519,1
...,...,...,...,...
14995,0.000000,0.659424,0.938788,10
14996,0.000000,0.703216,1.139663,10
14997,0.052274,1.003130,1.071929,10
14998,0.000000,0.941627,1.057998,10


In [44]:
import subprocess
# Assume df is your pandas DataFrame
df.to_csv('temp_data.csv', index=False)

# R script to convert CSV to RData
r_script = """
data <- read.csv('temp_data.csv')
save(data, file='data_matrix.RData')
"""

# Run the R script using subprocess
subprocess.run(['Rscript', '-e', r_script])

CompletedProcess(args=['Rscript', '-e', "\ndata <- read.csv('temp_data.csv')\nsave(data, file='data_matrix.RData')\n"], returncode=0)